# AgriVision pipeline on the CIA dataset (Landsat + MODIS)
Same modules as page 3 of `docs/diagrams/AgriVision_architecture.drawio`, run on the **Coleambally Irrigation
Area** benchmark (Emelyanova et al., CSIRO, DOI 10.4225/08/514B7FD1C798C): 17 cloud-free pairs, Oct 2001–May 2002,
Landsat-7 ETM+ at 25 m + MODIS MOD09GA (500 m, already resampled to the 25 m Landsat grid). Fine = Landsat here
(Sentinel-2 in your real pipeline), so the scale ratio is 20 (500 m / 25 m) instead of 25.

**Colab:** *Runtime → Change runtime type → GPU*, then give the notebook the data in **one** of two ways:
1. **Link:** on the CSIRO Data Access Portal page (open the DOI) click *Download*, copy the zip's link, paste it
   into `CIA_URL` below. Colab downloads it itself (fast).
2. **Drive:** if the portal only lets you download in the browser, put the zip (or the unzipped folder) anywhere
   in the top two levels of *My Drive*, with "CIA" or "Coleambally" in its name. The notebook finds it.

Then *Run all*.
CIA has MODIS only on the 17 pair dates, so "MODIS-only days" are the held-out test dates.

## 0 · Setup

In [ ]:
# installs the two packages Colab lacks
%pip install -q rasterio pytorch-msssim

In [ ]:
import os, re, zipfile, shutil, itertools, warnings, urllib.request, datetime as dt  # files, zips, warnings, downloads, dates
from pathlib import Path                                               # walking folders
import numpy as np, pandas as pd, matplotlib.pyplot as plt             # arrays, tables, plots
from scipy.ndimage import gaussian_filter, binary_dilation, label, shift as nd_shift  # image operations
from scipy.signal import savgol_filter                                 # time-series smoothing (M9)
from skimage.registration import phase_cross_correlation               # image matching (2.3)
from skimage.metrics import structural_similarity                      # SSIM score (M8)
import rasterio                                                        # raster read/write
from rasterio.transform import Affine                                  # georeference after cropping
import torch, torch.nn as nn, torch.nn.functional as F                 # neural network
from torch.utils.data import Dataset, DataLoader                       # batching training samples
from pytorch_msssim import ms_ssim                                     # MS-SSIM loss term
from IPython.display import display                                    # show tables

torch.manual_seed(42)                                                  # repeatable training
warnings.filterwarnings('ignore', category=RuntimeWarning)             # silence 'mean of empty slice' on no-data areas
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)  # raw files carry no georeference
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'                   # use the GPU when there is one
print('device:', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'CPU')  # confirm which one

CIA_URL = os.environ.get('CIA_URL', '')                                # paste the CSIRO zip link here (Colab downloads fast)
CIA_ZIP = os.environ.get('CIA_ZIP', '/content/drive/MyDrive/CIA.zip')  # ...or a zip/folder on Drive (searched for if not here)
WORK = os.environ.get('WORK', '/content/cia')                          # fast local disk for data and outputs
SAVE_TO_DRIVE = True                                                   # copy results to Drive at the end (Colab only)
os.makedirs(f'{WORK}/predicted', exist_ok=True)                        # where Module 7 writes GeoTIFFs

ROWS, COLS, NB = 2040, 1720, 6                                         # CIA image size and band count (CSIRO record)
RED, NIR = 2, 3                                                        # band order blue, green, red, NIR, SWIR1, SWIR2
K = 20                                                                 # 25 m Landsat pixels per 500 m MODIS pixel
EPOCHS, MAX_STEPS = int(os.environ.get('EPOCHS', 30)), int(os.environ.get('MAX_STEPS', 0)) or None  # training length; cap per epoch

---
## Module 1 · Data (CIA download)
Downloads and unzips CIA, finds every image, works out its sensor (`MOD` in the name = MODIS) and date (from
`A2001281`-style day-of-year, `20011008`-style dates, or `2001_281` folder names), and pairs Landsat with MODIS by
date (`modtran` in Landsat names is not mistaken for MODIS). Files with an ENVI header are read normally; headerless raw files are read as 16-bit and the byte order /
band interleave is detected by trying all six and keeping the one that gives a coherent (spatially smooth) image.

In [ ]:
HELP = ('Paste the CSIRO download link into CIA_URL, or put the CIA zip (or its unzipped folder) '
        'on Google Drive and set CIA_ZIP to its path.')                # what to do when the data is not found
if CIA_URL:                                                            # download straight into Colab
    urllib.request.urlretrieve(CIA_URL, f'{WORK}/CIA.zip'); CIA_ZIP = f'{WORK}/CIA.zip'  # download the zip; use it from now on
    assert zipfile.is_zipfile(CIA_ZIP), 'the link returned a web page, not a zip: ' + HELP  # portal links can need a login
elif CIA_ZIP.startswith('/content/drive'):                             # data on Google Drive
    from google.colab import drive; drive.mount('/content/drive')      # mount Google Drive
    if not os.path.exists(CIA_ZIP):                                    # not at that path: look for it (top 2 levels of Drive)
        # a zip or folder whose name contains the word CIA or Coleambally
        name_ok = lambda p: re.search(r'(^|[^a-z])cia([^a-z]|$)|coleambally', p.name, re.I) and (p.is_dir() or p.suffix.lower() == '.zip')
        found = [p for g in ('*', '*/*') for p in Path('/content/drive/MyDrive').glob(g) if name_ok(p)]  # zips/folders named CIA
        if not found: raise FileNotFoundError(f'CIA not found at {CIA_ZIP} or anywhere near the top of your Drive. ' + HELP)
        CIA_ZIP = str(found[0]); print('using', CIA_ZIP)               # first match
RAW = Path(f'{WORK}/raw')                                              # extracted files go here (fast local disk)
if os.path.isdir(CIA_ZIP): shutil.copytree(CIA_ZIP, RAW, dirs_exist_ok=True)  # already unzipped: just copy it over
else: zipfile.ZipFile(CIA_ZIP).extractall(RAW)                         # unzip the download
while inner := list(RAW.rglob('*.zip')):                               # the portal may nest zips: open those too
    for z in inner: zipfile.ZipFile(z).extractall(z.parent); z.unlink()  # extract next to it, then delete it
print(sum(p.is_file() for p in RAW.rglob('*')), 'files extracted')   # how many files arrived

In [ ]:
def find_date(path):                                                   # date written in a file or folder name
    s = str(path.relative_to(RAW))                                     # path inside the dataset: folder names count too
    if m := re.search(r'A((?:19|20)\d\d)(\d{3})', s):                  # MODIS style: A + year + day-of-year
        return dt.date(int(m[1]), 1, 1) + dt.timedelta(int(m[2]) - 1)  # 1 Jan + (day-of-year − 1)
    for m in re.finditer(r'((?:19|20)\d\d)(\d\d)(\d\d)', s):           # yyyymmdd anywhere
        try: return dt.date(int(m[1]), int(m[2]), int(m[3]))          # first one that is a real date
        except ValueError: pass  # not a real date: keep looking
    if m := re.search(r'((?:19|20)\d\d)[_-](\d{3})', s):               # folder style: year_dayofyear
        return dt.date(int(m[1]), 1, 1) + dt.timedelta(int(m[2]) - 1)  # 1 Jan + (day-of-year − 1)

# band-interleave options → (bands, rows, cols)
layouts = {'bil': lambda a: a.reshape(ROWS, NB, COLS).transpose(1, 0, 2), 'bsq': lambda a: a.reshape(NB, ROWS, COLS),
           'bip': lambda a: a.reshape(ROWS, COLS, NB).transpose(2, 0, 1)}  # band-interleave options → (bands, rows, cols)
coherence = lambda a: np.corrcoef(a[RED, ::4, :-1].ravel(), a[RED, ::4, 1:].ravel())[0, 1]  # neighbour correlation (real image ≈ 1)

def read_image(path):                                                  # → (red, NIR) reflectance, transform, crs
    try:                                                               # file with a header GDAL understands
        with rasterio.open(path) as f:  # open it
            a, tf, crs = f.read([RED + 1, NIR + 1]).astype(np.float32), f.transform, f.crs  # red + NIR bands, georeference
    except rasterio.errors.RasterioIOError:                            # headerless raw 16-bit file
        raw = np.fromfile(path, np.uint8)                              # bytes
        assert raw.size == ROWS * COLS * NB * 2, f'unexpected size {raw.size} for {path}'  # must be 2040 × 1720 × 6 × 2 bytes
        # all 6 readings: 2 byte orders × 3 interleaves
        options = [layouts[il](raw.view(bo)).astype(np.float32) for bo, il in itertools.product(('<i2', '>i2'), layouts)]
        a, tf, crs = max(options, key=coherence)[[RED, NIR]], None, None   # the reading that looks like an image
    a[a <= -1000] = np.nan                                             # -9999 = no data
    return a / (1e4 if np.nanmax(a) > 2 else 1), tf, crs               # stored as reflectance × 10000

# every raster file in the download
files = [p for p in RAW.rglob('*') if p.suffix.lower() in ('.bil', '.int', '.tif', '.tiff', '.img', '.dat', '.bsq')]
images = [p for p in files if not re.search('mask|paddock', str(p), re.I)]  # skip the irrigated-paddock mask
is_modis = lambda p: bool(re.search(r'(MOD|MYD|MCD)\d\d|MODIS', p.name, re.I))  # MODIS product codes (not 'modtran')
table = pd.DataFrame([dict(path=p, date=find_date(p), sensor='MODIS' if is_modis(p) else 'Landsat') for p in images])  # sensor + date per file
pairs = table.pivot_table(index='date', columns='sensor', values='path', aggfunc='first').dropna()  # dates with both
display(pairs.map(lambda p: Path(p).name))  # check: one Landsat and one MODIS file per date
# stop here if pairing failed
assert {'Landsat', 'MODIS'} <= set(pairs.columns) and len(pairs) > 5, 'pairing failed: check the table above and the file names'

In [ ]:
DATES = list(pairs.index)                                              # pair dates in order
day = {d: (d - DATES[0]).days for d in DATES}                          # date → day number since the first pair
land, modis, TF, CRS = {}, {}, None, None                              # images by day number; georeference
for d in DATES:  # every pair date
    land[day[d]], tf, crs = read_image(pairs.loc[d, 'Landsat'])        # fine image (Landsat)
    modis[day[d]], _, _ = read_image(pairs.loc[d, 'MODIS'])            # coarse image (MODIS, already on 25 m)
    TF, CRS = TF or tf, CRS or crs                                     # keep the first georeference found
D = sorted(land)                                                       # day numbers of all pairs
print({str(x): round(float(np.nanmean((land[day[x]][1] - land[day[x]][0]) / (land[day[x]][1] + land[day[x]][0]))), 2) for x in DATES})  # mean NDVI per date

The line above prints the mean NDVI of each Landsat date. Values around 0.2–0.6 confirm the band order
(red = band 3, NIR = band 4). If they are negative, red and NIR are swapped: change `RED, NIR` in Setup.

**Crop to the valid area:** CIA images have no-data borders. Start from the whole image and shave off whichever
edge has the most missing data until less than 0.1 % is missing, then trim to a multiple of 20 (whole 500 m cells).

In [ ]:
valid = np.all([np.isfinite(a[0]) for a in [*land.values(), *modis.values()]], axis=0)  # data in every image
r0, r1, c0, c1 = 0, ROWS, 0, COLS                                      # start with the whole image
while (bad := ~valid[r0:r1, c0:c1]).mean() > .001:                     # until under 0.1 % is missing
    edge = int(np.argmax([bad[0].mean(), bad[-1].mean(), bad[:, 0].mean(), bad[:, -1].mean()]))  # emptiest edge
    r0, r1, c0, c1 = r0 + (edge == 0), r1 - (edge == 1), c0 + (edge == 2), c1 - (edge == 3)  # shave it off
H, W = (r1 - r0) // K * K, (c1 - c0) // K * K                          # size, a multiple of 20
crop = lambda a: a[:, r0:r0 + H, c0:c0 + W].copy()                     # cut one image
land, modis = {d: crop(a) for d, a in land.items()}, {d: crop(a) for d, a in modis.items()}  # crop every image
TF = TF * Affine.translation(c0, r0) if TF else None                   # georeference of the cropped grid
HC, WC = H // K, W // K                                                # size of the 500 m grid
print(f'cropped to {H} × {W} px ({H * 25 / 1000:.1f} × {W * 25 / 1000:.1f} km)')  # final scene size

---
## Module 2 · Preprocessing (reflectance kept, no NDVI)
CIA is already cloud-free, atmospherically corrected, and MODIS is already resampled onto the Landsat grid.
So **2.1, 2.2, 2.4 and 2.5 reduce to no-data masking and a grid check**. **2.3 (co-registration), 2.6 (PSF + offset)
and 2.7 (band adjustment) still matter** and run as in the real pipeline.

**2.1 / 2.4 Masks:** no cloud layers ship with CIA; no-data (-9999) was already set to NaN when reading.
**2.2 / 2.5 Grid:** both sensors share one 25 m grid; check it.

In [ ]:
assert all(a.shape == (2, H, W) for a in [*land.values(), *modis.values()]), 'images are not on one grid'  # one grid
clear = {d: np.isfinite(land[d][0]).mean() for d in D}                 # valid fraction per Landsat date
modis_ok = {d: np.isfinite(modis[d][0]).mean() for d in D}             # valid fraction per MODIS date
coarse = lambda a: a.reshape(2, HC, K, WC, K).mean((2, 4))             # 25 m → 500 m block average
modis_500 = {d: coarse(modis[d]) for d in D}                           # MODIS back at its own scale (for 2.6, 2.7)

**2.3 Co-registration:** match each Landsat date's NIR to the clearest date (phase correlation) and shift it back.

In [ ]:
fill = lambda a: np.where(np.isfinite(a), a, np.nanmean(a))            # gaps → mean (matching needs complete images)

def shift_img(img, s):                                                 # shift by s pixels; gaps move with the image
    out = np.stack([nd_shift(np.nan_to_num(b), s, order=1, mode='nearest') for b in img])  # shift values
    out[:, nd_shift(np.isfinite(img[0]) * 1., s, order=1) < .999] = np.nan  # shift the valid mask too
    return out  # the result

MASTER = max(D, key=clear.get)                                         # clearest date is the reference
for d in D:                                                            # every date except the master
    if d != MASTER: land[d] = shift_img(land[d], phase_cross_correlation(fill(land[MASTER][1]), fill(land[d][1]), upsample_factor=20)[0])  # measure shift on NIR, undo on both bands

**2.6 PSF + offset fit:** find the blur and offset that make Landsat, blurred and averaged to 500 m, best match MODIS,
per date. **2.7 Band adjustment:** regress degraded Landsat on MODIS per band (`Landsat = a·MODIS + b`).

In [ ]:
def fit_psf_offset(fine, mod_img):                                     # search blur × offset for the best correlation
    best, ok = (-1, 0, 0, 0), np.isfinite(mod_img)                     # (r, sigma, dy, dx); usable MODIS cells
    for s in (0, 4, 8, 12, 16, 20):                                    # candidate blur widths (fine px)
        g = np.pad(gaussian_filter(fill(fine), s), 20, mode='edge')    # blurred Landsat, padded so it can be shifted
        # shift by (dy, dx), average to 500 m, correlate with MODIS
        score = lambda dy, dx: np.corrcoef(g[20 - dy:20 - dy + H, 20 - dx:20 - dx + W].reshape(HC, K, WC, K).mean((1, 3))[ok], mod_img[ok])[0, 1]
        _, dy, dx = max((score(a, b), a, b) for a in range(-20, 21, 4) for b in range(-20, 21, 4))  # coarse search
        # refine ±3 px around it and keep the overall best
        best = max(best, max((score(a, b), s, a, b) for a in range(dy - 3, dy + 4) for b in range(dx - 3, dx + 4) if max(abs(a), abs(b)) <= 20))
    return dict(zip(('r', 'sigma', 'dy', 'dx'), best))  # best correlation, blur and offset

fit = {d: fit_psf_offset(land[d][1], modis_500[d][1]) for d in D}      # every CIA date is a clear pair
display(pd.DataFrame(fit).T.round(3))                                  # fitted blur / offset per date

def degrade(fine, f):                                                  # Landsat as MODIS would see it
    # blur with the fitted sigma and shift by the fitted offset
    g = np.stack([nd_shift(gaussian_filter(fill(b), f['sigma']), (f['dy'], f['dx']), order=1, mode='nearest') for b in fine])
    return coarse(g)                                                   # average to 500 m

BAND_ADJ = []                                                          # (a, b) per band
for b in range(2):                                                     # red, then NIR
    x = np.concatenate([modis_500[d][b].ravel() for d in D])           # MODIS values on all dates
    y = np.concatenate([degrade(land[d], fit[d])[b].ravel() for d in D])  # matching degraded Landsat values
    ok = np.isfinite(x) & np.isfinite(y)                               # usable cells
    BAND_ADJ.append(np.polyfit(x[ok], y[ok], 1))                       # straight-line fit
print('band adjustment (a, b): red', BAND_ADJ[0].round(4), '| nir', BAND_ADJ[1].round(4))  # fitted coefficients

**Output of Module 2:** `land[d]` (co-registered) and `modis_fine(d)` (shifted back, band-adjusted), red + NIR
reflectance on the 25 m grid. No NDVI.

In [ ]:
_cache = {}                                                            # finished MODIS days, computed once
def modis_fine(d):                                                     # final 25 m MODIS for day d
    if d not in _cache:                                                # compute only the first time
        m = shift_img(modis[d], (-fit[d]['dy'], -fit[d]['dx']))         # undo the position error
        _cache[d] = np.stack([a * m[i] + c for i, (a, c) in enumerate(BAND_ADJ)]).astype(np.float32)  # band adjustment
    return _cache[d]                                                   # cached result

---
## Module 3 · Dataset
Samples **(C1, F1, C2) → F2** from pairs ≤ 32 days apart. Dates are split into train / validation / test, so test
dates are never seen. 256 px patches, stride 128, skipped if any image is > 5 % invalid; random flips/rotations.

In [ ]:
TEST = D[1:-1][1::3]                                                   # held-out dates (never first/last: need neighbours)
VAL = [d for d in D[1:-1][::3] if d not in TEST][:2]                   # validation dates
TRAIN = [d for d in D if d not in TEST + VAL]                          # training dates
date = lambda d: DATES[0] + dt.timedelta(days=int(d))                  # day number → calendar date
# the date split
print('train', [str(date(d)) for d in TRAIN], '\nval', [str(date(d)) for d in VAL], '\ntest', [str(date(d)) for d in TEST])

class Patches(Dataset):                                                # training samples: (C1, F1, C2) → F2 patches
    def __init__(self, targets, refs, augment):                        # every (reference, target) pair × patch position
        self.augment = augment                                         # flips/rotations on or off
        self.stack = lambda t1, t2: (modis_fine(t1), land[t1], modis_fine(t2), land[t2])  # C1, F1, C2, F2
        # pairs ≤ 32 days apart × patch positions (stride 128), kept if all four images are ≤ 5 % invalid
        self.items = [(t1, t2, i, j) for t2 in targets for t1 in refs if 0 < abs(t2 - t1) <= 32
                      for i in range(0, H - 255, 128) for j in range(0, W - 255, 128)  # ...patch positions...
                      if max(np.isnan(a[0, i:i + 256, j:j + 256]).mean() for a in self.stack(t1, t2)) <= .05]  # ...valid enough

    def __len__(self):                                                 # dataset size
        return len(self.items)  # number of samples

    def __getitem__(self, k):                                          # one sample
        t1, t2, i, j = self.items[k]                                   # which pair and where
        c1, f1, c2, f2 = (a[:, i:i + 256, j:j + 256] for a in self.stack(t1, t2))  # cut the patch
        x = np.stack([c1, np.where(np.isnan(f1), c1, f1), c2, np.where(np.isnan(f2), c2, f2)])  # rare gaps → MODIS value
        x = torch.from_numpy(np.nan_to_num(x)).clamp(0, 1)             # to tensor
        if self.augment:                                               # same random flip + 90° rotation for all four
            # random flip + random 90° rotation
            x = torch.rot90(x.flip(-1) if torch.rand(1) < .5 else x, int(torch.randint(4, (1,))), (-2, -1))
        return tuple(x)                                                # c1, f1, c2, f2

train_dl = DataLoader(Patches(TRAIN, TRAIN, True), batch_size=8, shuffle=True, drop_last=True)  # shuffled, augmented
val_dl = DataLoader(Patches(VAL, TRAIN, False), batch_size=8)          # unseen target dates, no augmentation
print(len(train_dl.dataset), 'training patches,', len(val_dl.dataset), 'validation patches')  # dataset sizes

---
## Module 4 · Fusion network (reflectance in → reflectance out)
Fine encoder (field detail from F1) + change encoder (fine change from [C1, F1, C2], last layer linear) → added
(fusion) → decoder (red, NIR).

In [ ]:
conv = lambda i, o: nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(i, o, 3))  # 3×3 conv that keeps the image size

class FusionNet(nn.Module):                                            # the fusion network
    def __init__(self, b=2):                                           # b = number of bands (red, NIR)
        super().__init__()                                             # PyTorch module setup
        # fine encoder: F1 → 128 maps of field detail
        self.fine_enc = nn.Sequential(conv(b, 32), nn.ReLU(), conv(32, 64), nn.ReLU(), conv(64, 128), nn.ReLU())
        # change encoder: [C1, F1, C2] → 128 maps of change (last layer linear: change can be negative)
        self.change_enc = nn.Sequential(conv(3 * b, 32), nn.ReLU(), conv(32, 64), nn.ReLU(), conv(64, 128))
        # decoder: 128 maps → red, NIR
        self.decoder = nn.Sequential(conv(128, 64), nn.ReLU(), conv(64, 32), nn.ReLU(), nn.Conv2d(32, b, 1))

    def forward(self, c1, f1, c2):                                     # predicted F2 = decode(detail + change)
        # fusion = add detail and change, then decode
        return self.decoder(self.fine_enc(f1) + self.change_enc(torch.cat([c1, f1, c2], 1)))

model = FusionNet().to(DEV)                                            # build it on the GPU
print(sum(p.numel() for p in model.parameters()), 'parameters')        # model size

---
## Modules 5–6 · Loss & training
L1 + 0.8·(1 − MS-SSIM) + 0.5·L1 on NDVI computed from the predicted bands (training signal only).
Adam, lr 1e-3, halved when validation stalls; the best epoch is kept (also copied to Drive at the end).

In [ ]:
ndvi_t = lambda x: (x[:, 1] - x[:, 0]) / (x[:, 1] + x[:, 0] + 1e-6)    # NDVI of a batch of predictions

def loss_fn(p, t):                                                     # pixel + structure + NDVI terms
    # L1 + 0.8·(1 − MS-SSIM) + 0.5·NDVI L1
    return F.l1_loss(p, t) + .8 * (1 - ms_ssim(p.clamp(0, 1), t.clamp(0, 1), data_range=1)) + .5 * F.l1_loss(ndvi_t(p), ndvi_t(t))

opt = torch.optim.Adam(model.parameters(), lr=1e-3)                    # optimiser (Paper 1's setting)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=.5, patience=4)  # halve lr when stuck
best = np.inf                                                          # best validation so far
for ep in range(EPOCHS):                                               # each epoch
    model.train()                                                      # training mode
    for c1, f1, c2, f2 in itertools.islice(train_dl, MAX_STEPS):       # each training batch (all, or the first MAX_STEPS)
        opt.zero_grad()                                                # clear old gradients
        loss = loss_fn(model(c1.to(DEV), f1.to(DEV), c2.to(DEV)), f2.to(DEV))  # predict and score against real F2
        loss.backward(); opt.step()                                    # update weights
    model.eval()                                                       # evaluation mode
    with torch.no_grad():                                              # mean validation L1 on unseen dates
        # mean L1 over validation batches
        val = np.mean([F.l1_loss(model(c1.to(DEV), f1.to(DEV), c2.to(DEV)), f2.to(DEV)).item() for c1, f1, c2, f2 in val_dl])
    sched.step(val)                                                    # maybe lower the learning rate
    if val < best: best = val; torch.save(model.state_dict(), f'{WORK}/best.pt')  # keep the best weights
    print(f'epoch {ep + 1}: train loss {loss.item():.4f} | val L1 {val:.4f}')  # progress
_ = model.load_state_dict(torch.load(f'{WORK}/best.pt', map_location=DEV))  # use the best epoch from here on

---
## Module 7 · Inference
Each test date is treated as a MODIS-only day: nearest train/val date as reference (F1, C1) + its MODIS (C2),
predicted in 256 px tiles with 32 px overlap. Optional per-band blend with a time guess, then GeoTIFFs.

In [ ]:
POOL = sorted(TRAIN + VAL)                                             # dates whose Landsat is used as reference
DAYS = TEST                                                            # "MODIS-only" days to predict
nearest = lambda d, pool: min(pool, key=lambda r: (abs(r - d), r > d)) # closest reference date (earlier wins ties)
filled = lambda d: np.where(np.isnan(land[d]), modis_fine(d), land[d]) # Landsat with rare gaps patched by MODIS

def predict(c1, f1, c2, tile=256, pad=32):                             # whole scene, tile by tile
    step = tile - 2 * pad                                              # kept centre of each tile
    nh, nw = -(-H // step) * step, -(-W // step) * step                # padded scene size
    # C1, F1, C2 padded (reflect) and moved to the GPU
    x = [torch.from_numpy(np.pad(np.nan_to_num(a), ((0, 0), (pad, nh - H + pad), (pad, nw - W + pad)), mode='reflect'))[None].float().to(DEV) for a in (c1, f1, c2)]
    out = np.zeros((2, nh, nw), np.float32)                            # stitched result
    with torch.no_grad():                                              # no gradients needed
        for i in range(0, nh, step):                                   # tile rows
            for j in range(0, nw, step):                               # tile columns: predict, keep the centre
                # predict a 256 px tile, keep its 192 px centre
                out[:, i:i + step, j:j + step] = model(*[a[..., i:i + tile, j:j + tile] for a in x])[0, :, pad:-pad, pad:-pad].cpu().numpy()
    return out[:, :H, :W].clip(0, 1)                                   # crop the padding

def fuse(d, pool=POOL):                                                # fusion prediction for day d
    r = nearest(d, pool)                                               # reference date
    out = predict(modis_fine(r), filled(r), modis_fine(d))             # C1, F1, C2 → F2
    out[:, np.isnan(modis_fine(d)[0])] = np.nan                        # no MODIS there → no prediction
    return out  # the result

def time_guess(d, pool=POOL):                                          # linear interpolation in time, per pixel
    a = max([r for r in pool if r <= d], default=min(pool))            # reference date before (or first)
    b = min([r for r in pool if r >= d], default=max(pool))            # reference date after (or last)
    w = 0 if a == b else (d - a) / (b - a)                             # how far between them
    return (1 - w) * filled(a) + w * filled(b)                         # weighted mix of the two images

E = np.zeros((3, 2, H, W))                                             # sums of eS², eT², eS·eT per pixel and band
for v in VAL:                                                          # hide each validation date and guess it
    pool = [r for r in POOL if r != v]                                 # references without it
    eS, eT = np.nan_to_num(land[v] - time_guess(v, pool)), np.nan_to_num(land[v] - fuse(v, pool))  # both errors
    E += [eS ** 2, eT ** 2, eS * eT]                                   # add this date's error products
eSS, eTT, eST = (np.stack([gaussian_filter(b, 5) for b in e]) for e in E)  # smooth the error statistics
W_S = np.clip((eTT - eST) / (eSS + eTT - 2 * eST + 1e-12), 0, 1).astype(np.float32)  # optimal weight on time guess
USE_BLEND = True                                                       # set False to output the model alone
# average weights per band
print('mean weight on time guess', W_S.mean((1, 2)).round(2), '| on model', (1 - W_S).mean((1, 2)).round(2))

model_only = {}                                                        # raw model output (for Module 8)
for d in DAYS:                                                         # every day to predict
    out = model_only[d] = fuse(d)                                      # model prediction
    final = np.where(np.isnan(out), np.nan, W_S * time_guess(d) + (1 - W_S) * out) if USE_BLEND else out  # blended image
    # GeoTIFF: red + NIR, reflectance × 10000 as int16, georeferenced when CIA provided a georeference
    with rasterio.open(f'{WORK}/predicted/{date(d)}.tif', 'w', driver='GTiff', height=H, width=W, count=2, dtype='int16',
                       # ...georeference, -9999 = no data, compressed
                       crs=CRS, transform=TF or Affine.identity(), nodata=-9999, compress='deflate') as f:
        f.write(np.where(np.isnan(final), -9999, np.round(final * 1e4)).astype(np.int16))  # write red, NIR

def read_pred(d):                                                      # GeoTIFF → reflectance (NaN = no data)
    with rasterio.open(f'{WORK}/predicted/{date(d)}.tif') as f:        # open the day's GeoTIFF
        a = f.read().astype(np.float32)                                # read both bands
    return np.where(a == -9999, np.nan, a / 1e4)                       # back to reflectance
print(len(DAYS), 'GeoTIFFs written to', f'{WORK}/predicted')          # summary

---
## Module 8 · Evaluation (held-out real Landsat dates)
Image metrics (RMSE, SSIM, SAM, ERGAS) and NDVI from each image, against reuse-F1, time interpolation and ESTARFM.

In [ ]:
def estarfm(F1, C1, F2, C2, Cp, win=51, step=5):                       # two reference pairs + coarse at prediction date
    T = lambda a: torch.tensor(np.nan_to_num(a), dtype=torch.float32, device=DEV)  # numpy → GPU tensor
    F1, C1, F2, C2, Cp = map(T, (F1, C1, F2, C2, Cp)); r = win // 2    # inputs on the GPU; window radius
    Fs, Cs = torch.cat([F1, F2]), torch.cat([C1, C2])                  # fine / coarse stacks: (date × band, H, W)
    thr = (2 * Fs.flatten(1).std(1) / 4).view(-1, 1, 1)                # similarity threshold per band (4 classes)
    fm, cm = Fs - Fs.mean(0), Cs - Cs.mean(0)                          # centred spectra
    R = (fm * cm).sum(0, keepdim=True) / (fm.norm(dim=0) * cm.norm(dim=0) + 1e-9)  # fine-coarse correlation per pixel
    P = [F.pad(a[None], (r,) * 4, mode='reflect')[0] for a in (Fs, Cs, R, Cp - C1, Cp - C2)]  # padded for window shifts
    wsum = d1 = d2 = n = sx = sy = sxy = sxx = 0                       # running sums over the window
    for oy in range(-r, r + 1, step):                                  # candidate rows in the window
        for ox in range(-r, r + 1, step):                              # candidate columns
            Fj, Cj, Rj, D1, D2 = (a[:, r + oy:r + oy + H, r + ox:r + ox + W] for a in P)  # neighbour values
            sim = ((Fj - Fs).abs() <= thr).all(0, keepdim=True).float()  # similar on both dates and bands?
            w = sim / ((1 - Rj).clamp(min=1e-3) * (1 + (oy ** 2 + ox ** 2) ** .5 / r))  # weight: purer and closer = more
            wsum, d1, d2 = wsum + w, d1 + w * D1, d2 + w * D2           # weighted coarse change since each reference
            x, y = Cj.view(2, 2, H, W), Fj.view(2, 2, H, W)            # (date, band) for the regression
            n, sx, sy = n + 2 * sim, sx + sim * x.sum(0), sy + sim * y.sum(0)  # regression sums: count, Σcoarse, Σfine
            sxy, sxx = sxy + sim * (x * y).sum(0), sxx + sim * (x * x).sum(0)  # regression sums: Σcoarse·fine, Σcoarse²
    var = sxx - sx ** 2 / n                                            # coarse variance among similar pixels
    V = torch.where((n >= 6) & (var > 1e-6), (sxy - sx * sy / n) / var.clamp(min=1e-9), torch.ones_like(var)).clamp(.2, 3)  # conversion coefficient
    box = lambda a: F.avg_pool2d(a[None], win, 1, r, count_include_pad=False)[0].sum(0, keepdim=True)  # window mean
    t1, t2 = 1 / ((box(C1) - box(Cp)).abs() + 1e-6), 1 / ((box(C2) - box(Cp)).abs() + 1e-6)  # trust the less-changed date
    # predict from each reference date, blend by temporal weight
    return ((t1 * (F1 + V * d1 / wsum) + t2 * (F2 + V * d2 / wsum)) / (t1 + t2)).clamp(0, 1).cpu().numpy()

ndvi = lambda x: (x[1] - x[0]) / (x[1] + x[0] + 1e-6)                  # NDVI of an image

def scores(p, t):                                                      # all metrics, on pixels valid in both
    ok = np.isfinite(t[0]) & np.isfinite(p[0]); P, T = p[:, ok], t[:, ok]  # valid pixels; predicted and real values
    cos = (P * T).sum(0) / (np.linalg.norm(P, axis=0) * np.linalg.norm(T, axis=0) + 1e-12)  # spectral angle cosine
    nP, nT = ndvi(P), ndvi(T)                                          # NDVI from predicted and real
    return dict(RMSE=np.sqrt(((P - T) ** 2).mean()),                   # reflectance error
                SSIM=np.mean([structural_similarity(np.where(ok, p[b], 0), np.where(ok, t[b], 0), data_range=1) for b in range(2)]),  # structure
                SAM=np.degrees(np.arccos(np.clip(cos, -1, 1))).mean(),  # spectral angle (degrees)
                ERGAS=100 / K * np.sqrt(((np.sqrt(((P - T) ** 2).mean(1)) / T.mean(1)) ** 2).mean()),  # relative global error
                NDVI_RMSE=np.sqrt(((nP - nT) ** 2).mean()), NDVI_R2=1 - ((nP - nT) ** 2).sum() / ((nT - nT.mean()) ** 2).sum())  # NDVI error, R²

rows = []                                                              # one row per (date, method)
for d in TEST:                                                         # every held-out date
    a, b = max(r for r in POOL if r < d), min(r for r in POOL if r > d)  # ESTARFM's reference dates around d
    # each method's image for this date
    preds = {'Fusion + blend (final GeoTIFF)': read_pred(d), 'Fusion model': model_only[d], 'Reuse F1': land[nearest(d, POOL)],
             'Time interpolation': time_guess(d), 'ESTARFM': estarfm(filled(a), modis_fine(a), filled(b), modis_fine(b), modis_fine(d))}  # each method
    rows += [dict(method=k, **scores(v, land[d])) for k, v in preds.items()]  # score each against real Landsat
display(pd.DataFrame(rows).groupby('method').mean().sort_values('NDVI_RMSE').round(4))  # best NDVI first

d = TEST[len(TEST) // 2]                                               # one test date to look at
fig, axs = plt.subplots(1, 4, figsize=(16, 4))                         # 4 panels
# real vs predicted vs baselines
for ax, (name, img) in zip(axs, {'real Landsat': land[d], 'fusion + blend': read_pred(d), 'reuse F1': land[nearest(d, POOL)], 'MODIS': modis_fine(d)}.items()):
    ax.imshow(ndvi(img)[:400, :400], vmin=0, vmax=.9, cmap='RdYlGn'); ax.set_title(f'{name}, NDVI {date(d)}'); ax.axis('off')  # 10 km zoom
plt.show()                                                             # draw

---
## Module 9 · NDVI & crop monitoring (LAST STEP)
NDVI from real Landsat (train/val dates) and from the predicted images (test dates) → cube flagged real/fused →
interpolated to a daily series and Savitzky-Golay smoothed per pixel, at 100 m (4 × 4 Landsat pixels) to fit in
Colab memory. Paddock statistics use CIA's irrigated-paddock mask when present.

In [ ]:
source = {d: 'real' if d in POOL else 'fused' for d in D}              # which dates are fused
cube = np.stack([ndvi(read_pred(d) if source[d] == 'fused' else land[d]) for d in D])  # (dates, H, W) NDVI
cube = np.nanmean(cube.reshape(len(D), H // 4, 4, W // 4, 4), (2, 4))  # 100 m cells (ignoring no-data pixels)
days = np.arange(D[-1] + 1)                                            # every day of the season
flat = cube.reshape(len(D), -1)                                        # (dates, cells)
fillgap = lambda f: np.where(np.isfinite(f), f, np.nanmean(f) if np.isfinite(f).any() else 0)  # missing dates → cell mean (0 if never seen)
daily = np.stack([np.interp(days, D, fillgap(f)) for f in flat.T], 1)  # daily values per cell
smooth = savgol_filter(daily, 31, 2, axis=0).astype(np.float32).reshape(len(days), H // 4, W // 4)  # 31-day, 2nd-order smoothing
# how many real / fused dates
print({k: sum(v == k for v in source.values()) for k in ('real', 'fused')}, 'dates;', len(days), 'days smoothed')

In [ ]:
mask_file = next((p for p in files if re.search('mask|paddock', str(p), re.I)), None)  # CIA's irrigated-paddock mask
if mask_file is not None:                                              # paddock statistics
    try:                                                               # mask with a header
        with rasterio.open(mask_file) as f: full = f.read(1)  # read band 1
    except rasterio.errors.RasterioIOError:                            # headerless 8- or 16-bit mask
        # raw bytes → 8- or 16-bit image
        raw = np.fromfile(mask_file, np.uint8); full = (raw if raw.size == ROWS * COLS else raw.view('<i2')).reshape(ROWS, COLS)
    m = full[r0:r0 + H, c0:c0 + W] > 0                                 # paddock pixels in the cropped area
    pid, n_p = label(m[::4, ::4])                                      # each connected paddock gets an id (100 m)
    cnt = np.bincount(pid.ravel(), minlength=n_p + 1)                  # cells per paddock
    curves = np.stack([np.bincount(pid.ravel(), s.ravel(), n_p + 1) for s in smooth], 1) / np.maximum(cnt, 1)[:, None]  # mean NDVI per paddock per day
    big = [p for p in range(1, n_p + 1) if cnt[p] >= 10]               # paddocks of ≥ 10 cells (≥ 10 ha)
    med = np.median(curves[big], 0)                                    # typical irrigated-paddock curve
    drop = lambda x: x - np.array([x[max(0, i - 20):i + 1].max() for i in range(len(x))])  # fall below best of last 20 days
    rows = []                                                          # one row per paddock
    for p in big:                                                      # each paddock
        x = curves[p]; pk = int(x.argmax()); half = (x.min() + x.max()) / 2  # season curve, peak day, half-amplitude
        run = np.convolve((drop(x) < -.05) & (drop(med) > -.02), np.ones(5), 'valid') >= 5  # own 5-day decline, median steady
        # size, phenology dates, peak...
        rows.append(dict(paddock=p, ha=int(cnt[p]), green_up=date(np.argmax(x >= half)), peak=date(pk), peak_ndvi=round(x.max(), 2),
                         senescence=date(pk + np.argmax(x[pk:] < half)), decline_alert=date(np.argmax(run)) if run.any() else ''))  # phenology + alert
    report = pd.DataFrame(rows).set_index('paddock')                   # full paddock report
    display(report.sort_values('peak_ndvi', ascending=False).head(15)) # strongest paddocks
    # how many paddocks to look at
    print(int((report.decline_alert != '').sum()), 'of', len(report), 'paddocks show an unexplained NDVI decline')
else:                                                                  # no mask in the download
    print('no paddock mask found: skipping paddock statistics')  # nothing to do

plt.figure(figsize=(14, 4))                                            # season maps
for i, d in enumerate(D[::4]):                                         # every 4th date
    plt.subplot(1, len(D[::4]), i + 1); plt.imshow(smooth[d], vmin=0, vmax=.9, cmap='RdYlGn'); plt.axis('off')  # smoothed NDVI
    plt.title(f'{date(d)} ({source[d]})', fontsize=9)                  # date and real/fused
plt.show()                                                             # draw

In [ ]:
if SAVE_TO_DRIVE and os.path.exists('/content'):                       # Colab: keep results after the session ends
    from google.colab import drive; drive.mount('/content/drive')      # mount Drive (no-op if already mounted)
    shutil.copytree(f'{WORK}/predicted', '/content/drive/MyDrive/agrivision_cia/predicted', dirs_exist_ok=True)  # GeoTIFFs
    shutil.copy(f'{WORK}/best.pt', '/content/drive/MyDrive/agrivision_cia/best.pt')  # trained weights
    print('results copied to MyDrive/agrivision_cia')  # done